# 🧠 Simple RAG with LlamaIndex + Nebius Token Factory

The smallest useful **Retrieval-Augmented Generation** pipeline you can build.

Point it at a folder of documents, ask a question, get an answer grounded in that folder's contents.

**Workflow**

1. **Load** every document in a local directory with `SimpleDirectoryReader`.
2. **Embed & index** them into an in-memory `VectorStoreIndex` using a Nebius embedding model.
3. **Query** through the index's query engine — it retrieves the most relevant chunks and hands them to a Nebius-hosted LLM, which writes a grounded answer.

No local models, no vector database to run, no server to deploy. Just an API key.

## 1. Install dependencies

Run this once per environment. Restart the kernel afterwards if the imports below fail.

In [ ]:
%pip install -q llama-index llama-index-llms-nebius llama-index-embeddings-nebius python-dotenv

## 2. Set your Nebius API key

Grab a key from [Nebius Token Factory](https://dub.sh/nebius).

Two options — pick either one:

- **Recommended:** copy `.env.example` to `.env`, put your key in it, and the cell below will pick it up automatically.
- **Quick and dirty:** paste the key straight into the `NEBIUS_API_KEY` assignment below.

> ⚠️ If you paste the key inline, clear the cell output and *never* commit the notebook with the key still in it. `.env` is already listed in `.gitignore`.

In [ ]:
import os
import getpass

from dotenv import load_dotenv

load_dotenv()  # reads a local .env file if one exists

# Option A: leave this as-is and use a .env file / exported environment variable.
# Option B: replace the fallback below with your key, e.g. NEBIUS_API_KEY = "your_nebius_api_key"
NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY", "")

# Last resort: prompt for it interactively so the key never lands in the notebook file.
if not NEBIUS_API_KEY:
    NEBIUS_API_KEY = getpass.getpass("Enter your Nebius API key: ")

os.environ["NEBIUS_API_KEY"] = NEBIUS_API_KEY

print("API key loaded ✅" if NEBIUS_API_KEY else "No API key found ❌")

## 3. Imports

- `SimpleDirectoryReader` — reads a folder of files (`.md`, `.txt`, `.pdf`, `.docx`, …) into LlamaIndex `Document` objects.
- `VectorStoreIndex` — builds the in-memory vector index and gives us a query engine.
- `Settings` — LlamaIndex's global config object; we point it at the Nebius LLM and embedding model.
- `NebiusLLM` / `NebiusEmbedding` — the Nebius Token Factory integrations.

In [ ]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.nebius import NebiusEmbedding
from llama_index.llms.nebius import NebiusLLM

## 4. The whole pipeline in one function

`run_rag_completion()` is the entire app. Give it a directory and a question, get back an answer.

| Parameter | Default | What it does |
| --- | --- | --- |
| `document_dir` | — | Folder to index. Read recursively? Set `recursive=True` on the reader. |
| `query_text` | — | The question to answer. |
| `embedding_model` | `BAAI/bge-en-icl` | Turns text into vectors for retrieval. |
| `generative_model` | `deepseek-ai/DeepSeek-V3` | Writes the final answer from the retrieved chunks. |
| `similarity_top_k` | `5` | How many chunks to feed the LLM. Raise it for broader context, lower it for tighter, cheaper answers. |

In [ ]:
def run_rag_completion(
    document_dir: str,
    query_text: str,
    embedding_model: str = "BAAI/bge-en-icl",
    generative_model: str = "deepseek-ai/DeepSeek-V3",
    similarity_top_k: int = 5,
) -> str:
    """Index a directory of documents and answer a question grounded in them.

    Args:
        document_dir: Path to the folder of documents to index.
        query_text: The question to ask.
        embedding_model: Nebius-hosted embedding model used for retrieval.
        generative_model: Nebius-hosted LLM used to write the answer.
        similarity_top_k: Number of retrieved chunks passed to the LLM.

    Returns:
        The generated answer as a string.
    """
    # Point LlamaIndex's global settings at Nebius Token Factory.
    Settings.llm = NebiusLLM(api_key=NEBIUS_API_KEY, model=generative_model)
    Settings.embed_model = NebiusEmbedding(api_key=NEBIUS_API_KEY, model_name=embedding_model)

    # 1. Load every document in the directory.
    documents = SimpleDirectoryReader(document_dir).load_data()

    # 2. Chunk, embed and index them in memory.
    index = VectorStoreIndex.from_documents(documents)

    # 3. Retrieve the top-k relevant chunks and generate a grounded answer.
    response = index.as_query_engine(similarity_top_k=similarity_top_k).query(query_text)

    return str(response)

## 5. Ask a question

`./data` ships with two small sample documents so you can run this immediately.
Swap `document_dir` for your own folder and change `query_text` to whatever you want to know.

In [ ]:
document_dir = "./data"
query_text = "What is retrieval-augmented generation, and why use it instead of fine-tuning?"

answer = run_rag_completion(document_dir, query_text)
print(answer)

## 6. Reuse the index for follow-up questions

`run_rag_completion()` re-indexes on every call, which is fine for a handful of documents but wasteful
for anything larger. Build the index once and reuse the query engine instead.

In [ ]:
Settings.llm = NebiusLLM(api_key=NEBIUS_API_KEY, model="deepseek-ai/DeepSeek-V3")
Settings.embed_model = NebiusEmbedding(api_key=NEBIUS_API_KEY, model_name="BAAI/bge-en-icl")

documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=5)

print(f"Indexed {len(documents)} document(s).\n")

for question in [
    "What models does Nebius Token Factory serve?",
    "What are the three stages of a RAG pipeline?",
]:
    print(f"Q: {question}")
    print(f"A: {query_engine.query(question)}\n")

## 7. Inspect the sources

The reason to use RAG over a bare LLM is traceability — you can see exactly which chunks the answer
came from. Every response carries its `source_nodes` along with a similarity score.

In [ ]:
response = query_engine.query("What are the three stages of a RAG pipeline?")

print(f"Answer:\n{response}\n")
print("-" * 70)

for i, node in enumerate(response.source_nodes, start=1):
    file_name = node.metadata.get("file_name", "unknown")
    snippet = node.get_content().strip().replace("\n", " ")[:200]
    print(f"\n[{i}] {file_name}  (score: {node.score:.4f})")
    print(f"    {snippet}...")

## Where to go next

This notebook is deliberately the floor, not the ceiling. Natural next steps:

- **Persist the index** — `index.storage_context.persist("./storage")` so you don't re-embed on every run.
- **Swap in a real vector store** — Qdrant, Chroma, or pgvector instead of the in-memory store.
- **Add reranking** — retrieve 20 chunks, rerank them, keep the best 5.
- **Hybrid search** — combine dense vectors with BM25 keyword matching.
- **Tune chunking** — `Settings.chunk_size` and `Settings.chunk_overlap` matter more than most people expect.
- **Stream the answer** — `index.as_query_engine(streaming=True)` for a responsive UI.

---

Built with [LlamaIndex](https://www.llamaindex.ai/) and [Nebius Token Factory](https://dub.sh/nebius).